# Section 5.5 - Monte Carlo Robustness Analysis

This notebook builds the paper-ready figures for the Monte Carlo robustness section.

It answers:
1. Which methods have better average simulated outcomes?
2. Which methods are more stable across scenarios?
3. Which methods have higher downside risk?
4. Do the Monte Carlo results confirm or contradict the backtest?
5. Which methods look most robust overall under uncertainty?

Core idea:
- backtest = one realized path
- Monte Carlo = distribution of possible outcomes


In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

PROJECT_ROOT = Path.cwd()
OUTPUT_CANDIDATES = [
    PROJECT_ROOT / 'outputs' / 'data_exports' / 'final_experimental_setup',
    PROJECT_ROOT / 'outputs' / 'final_experimental_setup',
]
OUTPUT_DIR = next((path for path in OUTPUT_CANDIDATES if path.exists()), OUTPUT_CANDIDATES[0])
TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR = OUTPUT_DIR / 'paper_figures' / 'section_5_5'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Using output directory: {OUTPUT_DIR}')
print(f'Figures will be written to: {FIGURE_DIR}')

mc_summary = pd.read_csv(TABLE_DIR / 'mc_summary.csv')
mc_terminal = pd.read_csv(TABLE_DIR / 'mc_terminal_values.csv')
backtest = pd.read_csv(TABLE_DIR / 'backtest_summary.csv')

print(f'MC summary rows: {len(mc_summary)}')
print(f'MC terminal rows: {len(mc_terminal)}')


Using output directory: c:\Users\narro\Desktop\semestre\honores_actuaria\outputs\data_exports\final_experimental_setup
Figures will be written to: c:\Users\narro\Desktop\semestre\honores_actuaria\outputs\data_exports\final_experimental_setup\paper_figures\section_5_5
MC summary rows: 135
MC terminal rows: 270000


In [2]:
METHOD_ORDER = [
    'equal_weight',
    'naive_risk_parity',
    'markowitz',
    'hrp_recursive',
    'hca_deprado_ew_nrp',
]

PALETTE = {
    'equal_weight': '#4C78A8',
    'naive_risk_parity': '#72B7B2',
    'markowitz': '#F58518',
    'hrp_recursive': '#54A24B',
    'hca_deprado_ew_nrp': '#E45756',
}


def method_label(method: str) -> str:
    labels = {
        'equal_weight': 'Equal Weight (1/N)',
        'naive_risk_parity': 'Naive Risk Parity',
        'markowitz': 'Markowitz',
        'hrp_recursive': 'HRP Recursive',
        'hca_deprado_ew_nrp': 'HCA De Prado EW/NRP',
    }
    return labels.get(method, method)


def apply_paper_layout(fig: go.Figure, *, title: str, width: int = 1100, height: int = 650) -> None:
    fig.update_layout(
        title=title,
        template='plotly_white',
        width=width,
        height=height,
        font=dict(size=13),
        margin=dict(l=70, r=40, t=90, b=65),
        legend=dict(orientation='h', x=0, y=1.02, xanchor='left', yanchor='bottom'),
        paper_bgcolor='white',
        plot_bgcolor='white',
    )


def save_figure(fig: go.Figure, filename: str) -> None:
    path = FIGURE_DIR / filename
    pio.write_html(fig, file=path, include_plotlyjs='cdn', full_html=True)
    print(f'Saved: {path}')


## Representative Run

To keep Section 5.5 aligned with Sections 5.4 and 5.2, the Monte Carlo analysis uses the same representative run where `Markowitz` underperformed in the historical backtest.


In [3]:
run_rows = []
for run_id, group in backtest.groupby('run_id'):
    mw_row = group[group['method'] == 'markowitz']
    if mw_row.empty:
        continue
    mw_row = mw_row.iloc[0]
    best_row = group.loc[group['sharpe_ratio'].idxmax()]
    run_rows.append(
        {
            'run_id': run_id,
            'markowitz_sharpe': mw_row['sharpe_ratio'],
            'best_method': best_row['method'],
            'best_sharpe': best_row['sharpe_ratio'],
            'gap_to_best': best_row['sharpe_ratio'] - mw_row['sharpe_ratio'],
        }
    )

run_quality = pd.DataFrame(run_rows).sort_values(
    ['gap_to_best', 'markowitz_sharpe'], ascending=[False, True]
)
RUN_ID = run_quality.iloc[0]['run_id']
print('Selected representative run:', RUN_ID)
run_quality.head(10)


Selected representative run: multi_asset_etf_20200601_24m


,run_id,markowitz_sharpe,best_method,best_sharpe,gap_to_best
13,multi_asset_etf_20200601_24m,0.193524,equal_weight,2.546559,2.353035
14,multi_asset_etf_20200601_36m,0.401885,equal_weight,2.546559,2.144674
12,multi_asset_etf_20200601_12m,0.449346,equal_weight,2.546559,2.097213
5,djia_20200601_36m,1.536411,equal_weight,2.496653,0.960242
4,djia_20200601_24m,1.620199,equal_weight,2.496653,0.876454
23,sp100_style_top100_20200601_36m,1.896605,equal_weight,2.738005,0.841400
26,sp100_style_top100_20220103_36m,-0.957516,naive_risk_parity,-0.362061,0.595455
25,sp100_style_top100_20220103_24m,-0.940850,naive_risk_parity,-0.367365,0.573484
8,djia_20220103_36m,-0.894429,hca_deprado_ew_nrp,-0.344017,0.550412
7,djia_20220103_24m,-0.844620,hca_deprado_ew_nrp,-0.319377,0.525243


In [4]:
mc_run = mc_summary[mc_summary['run_id'] == RUN_ID].copy()
terminal_run = mc_terminal[mc_terminal['run_id'] == RUN_ID].copy()

mc_run['method_label'] = mc_run['method'].map(method_label)
terminal_run['method_label'] = terminal_run['method'].map(method_label)

mc_run = mc_run.sort_values('method', key=lambda s: s.map({m: i for i, m in enumerate(METHOD_ORDER)}))
terminal_run = terminal_run.sort_values('method', key=lambda s: s.map({m: i for i, m in enumerate(METHOD_ORDER)}))

mc_run[['method','mean_terminal_value','median_terminal_value','std_terminal_value','prob_loss']]


,method,mean_terminal_value,median_terminal_value,std_terminal_value,prob_loss
110,equal_weight,1.062211,1.054993,0.142997,0.3485
111,naive_risk_parity,1.075831,1.071204,0.094363,0.2060
112,markowitz,1.132543,1.130855,0.055234,0.0040
113,hrp_recursive,1.077805,1.074451,0.067209,0.1200
114,hca_deprado_ew_nrp,1.109486,1.108151,0.087375,0.1040


## Figure 1 - Terminal Value Distribution

This figure answers: who performs better on average under uncertainty, and how wide are the simulated outcome distributions?


In [5]:
fig_dist = go.Figure()

for method in METHOD_ORDER:
    subset = terminal_run[terminal_run['method'] == method]
    if subset.empty:
        continue
    fig_dist.add_trace(
        go.Histogram(
            x=subset['terminal_value'],
            name=method_label(method),
            marker_color=PALETTE[method],
            opacity=0.45,
            histnorm='probability density',
            nbinsx=50,
        )
    )

fig_dist.update_layout(barmode='overlay')
fig_dist.update_xaxes(title='Terminal portfolio value')
fig_dist.update_yaxes(title='Density')
apply_paper_layout(fig_dist, title=f'Monte Carlo Terminal Value Distribution - {RUN_ID}')
save_figure(fig_dist, 'figure_01_terminal_value_distribution.html')
fig_dist


Saved: c:\Users\narro\Desktop\semestre\honores_actuaria\outputs\data_exports\final_experimental_setup\paper_figures\section_5_5\figure_01_terminal_value_distribution.html


## Figure 2 - Method Comparison

This figure answers: which methods are more stable across scenarios, and which ones show wider or more asymmetric terminal-value distributions?


In [6]:
fig_box = px.box(
    terminal_run,
    x='method_label',
    y='terminal_value',
    color='method_label',
    category_orders={'method_label': [method_label(m) for m in METHOD_ORDER]},
    color_discrete_map={method_label(k): v for k, v in PALETTE.items()},
    points='outliers',
)
fig_box.update_xaxes(title='Method')
fig_box.update_yaxes(title='Terminal portfolio value')
apply_paper_layout(fig_box, title=f'Monte Carlo Terminal Value Comparison - {RUN_ID}', width=1050, height=620)
save_figure(fig_box, 'figure_02_terminal_value_boxplot.html')
fig_box


Saved: c:\Users\narro\Desktop\semestre\honores_actuaria\outputs\data_exports\final_experimental_setup\paper_figures\section_5_5\figure_02_terminal_value_boxplot.html


## Figure 3 - Probability of Loss

This figure answers: which methods have the highest downside risk under the simulated distribution?


In [7]:
prob_loss_df = mc_run.copy()
prob_loss_df['method_label'] = pd.Categorical(
    prob_loss_df['method_label'],
    categories=[method_label(m) for m in METHOD_ORDER],
    ordered=True,
)
prob_loss_df = prob_loss_df.sort_values('method_label')

fig_loss = px.bar(
    prob_loss_df,
    x='method_label',
    y='prob_loss',
    color='method_label',
    text='prob_loss',
    category_orders={'method_label': [method_label(m) for m in METHOD_ORDER]},
    color_discrete_map={method_label(k): v for k, v in PALETTE.items()},
)
fig_loss.update_traces(texttemplate='%{text:.1%}', textposition='outside')
fig_loss.update_xaxes(title='Method')
fig_loss.update_yaxes(title='Probability of loss', tickformat='.0%')
apply_paper_layout(fig_loss, title=f'Probability of Loss by Method - {RUN_ID}', width=1050, height=560)
save_figure(fig_loss, 'figure_03_probability_of_loss.html')
fig_loss


Saved: c:\Users\narro\Desktop\semestre\honores_actuaria\outputs\data_exports\final_experimental_setup\paper_figures\section_5_5\figure_03_probability_of_loss.html


## Writing Prompts

Use this flow in the paper:

1. Start with the distribution of terminal outcomes.
2. Then explain dispersion and stability.
3. Then discuss downside risk using the probability of loss.
4. Then compare the Monte Carlo evidence with the historical backtest.
5. End by stating which methods look robust overall under uncertainty.

One-line memory:

`5.5 = robustness beats performance`
